# Stage 4, tier 2: LucaOne representations and IsoFormer-style GTEx comparisons

This notebook develops two related Stage 4 questions. First, how well do LucaOne's frozen representations predict mRNA stability compared with BioLangFusion's original encoders and features calculated directly from the sequence? Second, how consistent are the local IsoFormer-style GTEx modality comparisons across additional training seeds? The experiments use different datasets and predictors, so they do not form a direct LucaOne-versus-IsoFormer comparison.

LucaOne's checkpoint was pretrained on nucleotide and protein sequences. Here, its weights remain frozen: we encode each coding sequence and its translated protein separately, average their token representations, and fit ridge regression to either vector or their concatenation. The same ridge procedure reads the existing BioLangFusion encoder vectors. Parts 1–3 establish the input scope, extraction cost and predictive comparison.

Part 4 returns to the GTEx pilot, where genomic DNA, transcript RNA and protein inputs cover different biological regions. It reuses frozen encoder outputs and trains the local IsoFormer-style cross-attention predictor with additional seeds. This asks about variation in the local fusion experiment; it does not reproduce the paper's end-to-end encoder fine-tuning.


The [Stage 4 guide](README.md#lucaone-comparing-jointly-pretrained-representations) connects these comparisons to the surrounding experiments. [Stage 3's LucaOne and IsoFormer checks](../stage3/04_lucaone_isoformer_check.ipynb) establish their local implementations; [Stage 3 token extraction](../stage3/02_token_embeddings.ipynb) supplies the BioLangFusion caches reused below. These inputs do not depend on completing the Phase B encoder swaps. The [research plan](../../docs/Research_Project_Plan_Gantt.xlsx), `Gantt` and `Details` tasks 41 and 44, records the two research aims.


---

## Setup

Prepare the environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies) and select that environment as the notebook kernel. Package installation belongs in that setup step.

### Imports


In [1]:
# Files, project paths, serialized checkpoint names and elapsed-time measurements.
import json
import os
import sys
import time
from pathlib import Path

# Enable CPU fallback for unsupported Apple GPU operations before importing torch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables.
import numpy as np
import pandas as pd

# Encoder inference, device selection and numerical precision.
import torch

# Ridge prediction, feature scaling and their shared fitting pipeline.
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Display width is measured in characters, not sequence positions.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print(
    "torch", torch.__version__,
    "| CUDA:", torch.cuda.is_available(),
    "| Apple GPU:", torch.backends.mps.is_available(),
)


torch 2.10.0+cu126 | CUDA: True | Apple GPU: False


### Project paths and shared code

Locate the project's `Code` directory before importing the shared helpers. Input and result locations below are anchored to this directory so they refer to the same files when the kernel starts elsewhere inside the repository.


In [ ]:
# Find Code/shared_code from the repository root or a folder beneath it.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Make shared_code importable without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Dataset preparation, frozen encoders and sequence-derived features.
from shared_code import datasets, encoders, lucaone, sequences

# Fusion configurations, GTEx inputs, prediction scores and saved results.
from shared_code.fusion import blf, isoformer, train
from shared_code.fusion import token_embeddings as tokens


### Settings, file locations and hardware

The selected benchmark is mRNA stability, and `CHECKPOINT` chooses the jointly pretrained LucaOne checkpoint. “Jointly pretrained” describes how the checkpoint learned; it does not mean that this notebook feeds the nucleotide and protein sequences into it together.

The device preference is NVIDIA CUDA, then Apple MPS, then CPU. LucaOne uses float16 on CUDA and its default float32 elsewhere. Half precision reduces weight storage, but weight bytes alone do not establish peak RAM or GPU memory: activations and other allocations also matter. The timing section reports the loaded model's parameter count; the earlier 1.8B family description is not the parameter count measured for every local load.

`RUN_LUCAONE` controls full embedding extraction or cache loading. It does **not** guard the earlier timing cell, which loads the model and performs inference. `RUN_ISOFORMER_SEEDS` controls the separate GTEx training block in Part 4. Inspect both settings before executing those sections.


In [ ]:
# Selected stability comparison and LucaOne checkpoint.
BENCHMARK = "mrna_stability"
CHECKPOINT = "default"

# Existing execution choices; the timing cell remains independent of these flags.
RUN_LUCAONE = True
RUN_ISOFORMER_SEEDS = False

# Bootstrap resamples the fixed test predictions; it does not refit the probe.
BOOTSTRAP_DRAWS = 4000
PROBE_SEEDS = (0, 1, 2)  # Retained legacy setting; no code consumes it here.

# Reuse Stage 3 embeddings and keep the two analyses' results in Stage 4.
STAGE3_DIR = CODE_DIR / "stage3"
STAGE4_DIR = CODE_DIR / "stage4"
CACHE_ROOT = STAGE3_DIR / "embeddings"
LUCAONE_DIR = CACHE_ROOT / BENCHMARK / "lucaone"
RESULTS_DIR = STAGE4_DIR / "results" / "lucaone"
GTEX_RESULTS = STAGE4_DIR / "results" / "gtex_isoformer"


In [12]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
DTYPE = torch.float16 if DEVICE == "cuda" else None
print(f"device {DEVICE}, LucaOne dtype {DTYPE or 'float32'}")
print(f"checkpoints published: {json.dumps(lucaone.CHECKPOINTS, indent=2)}")


device cuda, LucaOne dtype torch.float16
checkpoints published: {
  "default": "LucaGroup/LucaOne-default-step36M",
  "gene": "LucaGroup/LucaOne-gene-step36.8M",
  "prot": "LucaGroup/LucaOne-prot-step38.2M"
}


---

## Part 1: which inputs fit the selected LucaOne context?

For the stability benchmark, a row supplies a coding sequence and a numerical stability target. LucaOne's gene stream reads the nucleotide sequence; its protein stream reads the translated amino-acid sequence. Separate calls allow each input its own context. A joint input would place both sequences in one context, leaving less room for each.

The coverage helper checks the selected 1,280-token context, including framing tokens. For a complete coding sequence of length `L` nucleotides, a joint input needs `L + L // 3 + 3` tokens: nucleotides, translated residues and framing tokens. The largest complete coding sequence fitting that rule is 957 nucleotides. [The shared LucaOne helper](../shared_code/lucaone.py) implements this arithmetic.

The table also examines GTEx's genomic DNA windows, full transcripts (`RNA`) and coding regions (`CDS`). **Read the GTEx protein and joint columns cautiously:** the helper estimates protein length as nucleotide length divided by three. That assumption describes an in-frame coding sequence; it does not describe the actual protein encoded by a genomic window or a full transcript containing untranslated regions. Their gene-stream columns still measure nucleotide-context coverage.


In [13]:
coverage = []
for key in datasets.BLF_BENCHMARKS:
    rows = datasets.blf_benchmark_rows(key)
    coverage.append({
        "benchmark": key,
        **lucaone.coverage(rows["length"].to_numpy()),
    })

gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)
for column in ("DNA", "RNA", "CDS"):
    length = gtex[column].astype(str).str.strip().str.len()
    # Protein/joint estimates assume a CDS; see the GTEx qualification above.
    coverage.append({
        "benchmark": f"gtex_pilot/{column}",
        **lucaone.coverage(length.to_numpy()),
    })

table = pd.DataFrame(coverage).set_index("benchmark")
display(table.style.format({
    "gene stream": "{:.1%}",
    "protein stream": "{:.1%}",
    "joint input": "{:.1%}",
}))
print(
    f"The joint mode needs 4L/3 + 3 tokens, so it caps a coding sequence at "
    f"{lucaone.longest_joint_cds()} nt against LucaOne's "
    f"{lucaone.CONTEXT_TOKENS}-token context."
)


,rows,longest nt,gene stream,protein stream,joint input
benchmark,,,,,
fungal_expression,3138,999,100.0%,100.0%,93.7%
ecoli_proteins,4450,999,100.0%,100.0%,96.4%
mrna_stability,23929,999,100.0%,100.0%,93.4%
cov_vac_degradation,2400,81,100.0%,100.0%,100.0%
gtex_pilot/DNA,2000,6000,0.0%,0.0%,0.0%
gtex_pilot/RNA,2000,21116,39.9%,83.9%,31.6%
gtex_pilot/CDS,2000,16080,68.1%,96.8%,58.0%


The joint mode needs 4L/3 + 3 tokens, so it caps a coding sequence at 957 nt against LucaOne's 1280-token context.


**What this establishes.** The preprocessed BioLangFusion coding sequences fit separately within the selected context; stability is the benchmark chosen for this comparison, not the only BioLangFusion benchmark that fits. Some coding sequences exceed the joint-input limit, so the extraction below encodes nucleotide and protein inputs separately.

GTEx's 6,000-nucleotide DNA windows exceed the selected whole-input context. This notebook therefore omits the LucaOne GTEx comparison. That is a restriction of this comparison, not proof that LucaOne cannot process GTEx under any procedure: the local helper supports windowed pooling, but using it here would change the input treatment. The previously recorded checkpoint configuration has a larger position setting with position embeddings disabled; this comparison still uses the selected 1,280-token rule. The [Stage 4 guide](README.md#lucaone-comparing-jointly-pretrained-representations) retains this scope distinction.


---

## Part 2: timing and extracting LucaOne representations

Load the selected stability rows before preparing their two input streams. The length range below refers to the preprocessed coding sequences actually used by this comparison.


In [ ]:
rows = datasets.blf_benchmark_rows(BENCHMARK)


### Measure a small inference sample

This cell loads LucaOne, potentially downloads missing weights into the Hugging Face cache, and runs inference on the first eight gene sequences. It runs regardless of `RUN_LUCAONE`. It does not save sequence embeddings.

The projection multiplies the measured gene-input rate by the number of rows and doubles it for two streams. These first rows are not a representative random timing sample, and protein inputs need not cost the same as gene inputs. Treat the result as a rough estimate for that device and sample, not a completion-time guarantee or a peak-memory measurement.


In [14]:
print(
    f"{BENCHMARK}: {len(rows):,} rows, "
    f"{rows['length'].min()} to {rows['length'].max()} nt"
)

# The gene tokenizer maps T and U to the same nucleotide token.
PROBE_ROWS = 8
sample = rows["sequence"].head(PROBE_ROWS).tolist()
started = time.perf_counter()
tokenizer, model = lucaone.load(CHECKPOINT, DEVICE, DTYPE)
load_seconds = time.perf_counter() - started
parameters = sum(p.numel() for p in model.parameters())

started = time.perf_counter()
for text in sample:
    lucaone.pooled(text, "gene", tokenizer, model, DEVICE)
per_sequence = (time.perf_counter() - started) / len(sample)

del model, tokenizer
encoders.free_memory()

print(
    f"LucaOne {CHECKPOINT}: {parameters / 1e9:.2f}B parameters, "
    f"loaded in {load_seconds:.0f} s"
)
print(f"{per_sequence:.2f} s per sequence on {DEVICE}")
for streams, label in ((1, "gene stream only"), (2, "gene and protein streams")):
    hours = per_sequence * len(rows) * streams / 3600
    print(
        f"  {label}: {hours:.1f} hours for "
        f"{len(rows) * streams:,} forward passes"
    )


mrna_stability: 23,929 rows, 3 to 999 nt


Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LucaOne default: 1.57B parameters, loaded in 33 s
0.19 s per sequence on cuda
  gene stream only: 1.3 hours for 23,929 forward passes
  gene and protein streams: 2.5 hours for 47,858 forward passes


### Prepare and embed the two streams

The gene input is the benchmark's coding sequence. The protein input uses `grid_protein`, the translation also used by BioLangFusion's Phase A protein encoder: complete codons are translated and stop symbols become `X`. LucaOne produces token representations, which the local helper averages over content tokens to obtain one vector per input. “Both streams” later means concatenating those two separately obtained vectors.

Each stream has its own cache. Reuse checks include checkpoint name, stream type, context and an ordered-input fingerprint; they do not establish every aspect of the original extraction environment. Progress messages count processed sequences, but the cache is published only after the **whole stream** finishes. An interrupted partial stream must be recomputed; a completed stream can be reused.

The manifest is written for the current invocation, including when matrices are reused. Its device and precision fields should not be treated as proof of the cached matrices' original producing environment.


In [ ]:
gene_texts = rows["sequence"].tolist()
protein_texts = [tokens.grid_protein(s) for s in rows["sequence"]]


In [15]:
LUCAONE_DIR.mkdir(parents=True, exist_ok=True)

streams = {}
if RUN_LUCAONE:
    for name, texts in (("gene", gene_texts), ("prot", protein_texts)):
        started = time.perf_counter()
        matrix, reused = lucaone.embed_with_cache(
            texts,
            name,
            LUCAONE_DIR / f"{CHECKPOINT}_{name}.npz",
            device=DEVICE,
            checkpoint=CHECKPOINT,
            dtype=DTYPE,
        )
        streams[name] = matrix
        print(
            f"  {name}: {matrix.shape}, "
            f"{'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}"
        )

    # This records the invocation even when both matrices came from existing caches.
    lucaone.write_manifest(
        LUCAONE_DIR / "manifest.json",
        benchmark=BENCHMARK,
        checkpoint=lucaone.CHECKPOINTS[CHECKPOINT],
        rows=len(rows),
        device=DEVICE,
        dtype=str(DTYPE),
        context=lucaone.CONTEXT_TOKENS,
    )
else:
    print("RUN_LUCAONE is off. Read the projection above, then set it to True and rerun.")


Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    lucaone/gene: 200/23929
    lucaone/gene: 400/23929
    lucaone/gene: 600/23929
    lucaone/gene: 800/23929
    lucaone/gene: 1000/23929
    lucaone/gene: 1200/23929
    lucaone/gene: 1400/23929
    lucaone/gene: 1600/23929
    lucaone/gene: 1800/23929
    lucaone/gene: 2000/23929
    lucaone/gene: 2200/23929
    lucaone/gene: 2400/23929
    lucaone/gene: 2600/23929
    lucaone/gene: 2800/23929
    lucaone/gene: 3000/23929
    lucaone/gene: 3200/23929
    lucaone/gene: 3400/23929
    lucaone/gene: 3600/23929
    lucaone/gene: 3800/23929
    lucaone/gene: 4000/23929
    lucaone/gene: 4200/23929
    lucaone/gene: 4400/23929
    lucaone/gene: 4600/23929
    lucaone/gene: 4800/23929
    lucaone/gene: 5000/23929
    lucaone/gene: 5200/23929
    lucaone/gene: 5400/23929
    lucaone/gene: 5600/23929
    lucaone/gene: 5800/23929
    lucaone/gene: 6000/23929
    lucaone/gene: 6200/23929
    lucaone/gene: 6400/23929
    lucaone/gene: 6600/23929
    lucaone/gene: 6800/23929
    lucaone/gene: 

Loading weights:   0%|          | 0/344 [00:00<?, ?it/s]

[transformers] LucaGPLMModel LOAD REPORT from: LucaGroup/LucaOne-default-step36M
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    lucaone/prot: 200/23929
    lucaone/prot: 400/23929
    lucaone/prot: 600/23929
    lucaone/prot: 800/23929
    lucaone/prot: 1000/23929
    lucaone/prot: 1200/23929
    lucaone/prot: 1400/23929
    lucaone/prot: 1600/23929
    lucaone/prot: 1800/23929
    lucaone/prot: 2000/23929
    lucaone/prot: 2200/23929
    lucaone/prot: 2400/23929
    lucaone/prot: 2600/23929
    lucaone/prot: 2800/23929
    lucaone/prot: 3000/23929
    lucaone/prot: 3200/23929
    lucaone/prot: 3400/23929
    lucaone/prot: 3600/23929
    lucaone/prot: 3800/23929
    lucaone/prot: 4000/23929
    lucaone/prot: 4200/23929
    lucaone/prot: 4400/23929
    lucaone/prot: 4600/23929
    lucaone/prot: 4800/23929
    lucaone/prot: 5000/23929
    lucaone/prot: 5200/23929
    lucaone/prot: 5400/23929
    lucaone/prot: 5600/23929
    lucaone/prot: 5800/23929
    lucaone/prot: 6000/23929
    lucaone/prot: 6200/23929
    lucaone/prot: 6400/23929
    lucaone/prot: 6600/23929
    lucaone/prot: 6800/23929
    lucaone/prot: 

---

## Part 3: compare representations with the same ridge predictor

The representations are now inputs to a smaller fitted predictor. Ridge regression learns a weighted combination of features while penalizing large coefficients. Applying the same predictor family and supplied partitions to each representation makes their predictive performance comparable under this procedure. Checkpoint capacity, vector width and precision still differ, so a score difference does not isolate joint pretraining as its cause.

### Prepare partitions and comparison inputs

`Value` is the stability target and `Split` identifies the supplied partitions. The predictor fits on training rows and is scored on test rows. The supplied validation rows are unused: `RidgeCV` selects its penalty internally using the training data. The original reason for choosing this procedure over the supplied validation partition is not established.

The BioLangFusion vectors are means of content-token representations from Stage 3's existing caches. Their row order must correspond to the current benchmark rows; opening these caches here is not a fresh verification of every input's identity. The calculated baseline has **87 features**: GC content, GC at the third codon position, log sequence length, 64 codon frequencies and 20 amino-acid frequencies. Its retained table label abbreviates that feature list. [Sequence feature code](../shared_code/sequences.py) defines the calculation.


In [ ]:
# Keep the supplied partitions; do not construct new folds or shuffle row order.
split = rows["Split"].to_numpy()
y = rows["Value"].to_numpy(dtype=float)
train_rows = np.flatnonzero(split == "train")
test_rows = np.flatnonzero(split == "test")


In [16]:
print(f"published split: {len(train_rows):,} train, {len(test_rows):,} test")

# Average existing content-token representations into one vector per input row.
pooled = {}
for modality, encoder_key in blf.ENCODER_OF_MODALITY.items():
    cache = tokens.summarize(CACHE_ROOT / BENCHMARK / encoder_key)
    pooled[encoders.ENCODERS[encoder_key].label] = tokens.pooled_matrix(cache)
    print(f"  {encoder_key}: {pooled[encoders.ENCODERS[encoder_key].label].shape}")

# Sequence-derived features include amino-acid composition as well as codon use.
composition = np.array([
    sequences.composition_features(s) for s in rows["sequence"]
])
print(f"  composition features: {composition.shape}")


published split: 16,742 train, 3,579 test
  ntv2_100m: (23929, 512)
  rnafm: (23929, 640)
  esm2_8m: (23929, 320)
  composition features: (23929, 87)


### Define the predictor and assemble representations

The pipeline learns feature scaling from training rows and then fits `RidgeCV` over the existing penalty grid. A fresh pipeline is constructed for every representation. Test rows are transformed using the fitted scaling and used only to obtain predictions and their Spearman rank correlation with the stability targets. The [linear-probing guide](../../docs/methods/linear-probing.md) explains why this measures what the selected predictor can recover from frozen representations.

Concatenation joins vectors for the same row along the feature axis. It neither updates the encoders nor introduces token-level cross-attention. Comparing an embedding predictor with the composition predictor is also different from fitting embeddings to residual errors after composition controls, as the mRFP notebook does.


In [ ]:
def probe_on_split(X, y, train_rows, test_rows):
    # Fit scaling and the ridge penalty using training rows, then predict test rows.
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 5, 20))),
    ])
    model.fit(X[train_rows], y[train_rows])
    predicted = model.predict(X[test_rows])
    return predicted, train.score_rows("regression", y[test_rows], predicted)


In [ ]:
representations = {
    "Composition (GC, GC3, codon use, length)": composition,
    **pooled,
    "BLF three encoders concatenated": np.hstack(list(pooled.values())),
}
if streams:
    representations["LucaOne, gene stream"] = streams["gene"]
    representations["LucaOne, protein stream"] = streams["prot"]
    representations["LucaOne, both streams"] = np.hstack([
        streams["gene"], streams["prot"],
    ])


Fit one predictor per representation and retain its test predictions for the paired comparison afterward. In the table, `width` counts input features per row; `probe Spearman` measures agreement between predicted and observed stability rankings. Neither column measures training speed or biological mechanism.


In [17]:
scores, predictions = {}, {}
for name, X in representations.items():
    predictions[name], scores[name] = probe_on_split(X, y, train_rows, test_rows)
    print(f"  {name}: Spearman {scores[name]:.4f}")

summary = pd.DataFrame({
    "probe Spearman": pd.Series(scores),
    "width": {k: v.shape[1] for k, v in representations.items()},
})
display(summary.sort_values("probe Spearman", ascending=False).round(4))


  Composition (GC, GC3, codon use, length): Spearman 0.2828
  NT v2 100M multi-species: Spearman 0.3124
  RNA-FM: Spearman 0.3083
  ESM-2 8M: Spearman 0.3092
  BLF three encoders concatenated: Spearman 0.3904
  LucaOne, gene stream: Spearman 0.4435
  LucaOne, protein stream: Spearman 0.4007
  LucaOne, both streams: Spearman 0.4828


,probe Spearman,width
"LucaOne, both streams",0.4828,5120
"LucaOne, gene stream",0.4435,2560
"LucaOne, protein stream",0.4007,2560
BLF three encoders concatenated,0.3904,1472
NT v2 100M multi-species,0.3124,512
ESM-2 8M,0.3092,320
RNA-FM,0.3083,640
"Composition (GC, GC3, codon use, length)",0.2828,87


**Reading the retained result.** The saved stability table places the two LucaOne vectors together above the concatenated BioLangFusion vectors under this ridge procedure. The [October 22 result record](../../docs/run-records.md#2026-10-22-stage4-human-probes-completed-half-life-translation-efficiency-and-lucaone-results) identifies that execution. This supports a predictive comparison of the selected representations; it does not establish that joint pretraining caused the gain or that the representation adds information conditionally beyond the calculated controls. The next cell estimates uncertainty in selected score differences.


### Compare score differences on the same test examples

The paired bootstrap resamples test sequence groups and scores every predictor on the same resample. Identical sequences in the test partition stay together. Each row reports the first predictor's Spearman correlation minus the second's; the interval describes variation in that difference under this resampling procedure.

The “best BLF single encoder” is the encoder with the highest **observed test score** in this run. That selection is not repeated within the bootstrap, so its interval is conditional on the selected comparison. Although the helper is named `seed_averaged_bootstrap`, this notebook supplies one prediction array per representation: it does not repeat fits or partition assignments. `PROBE_SEEDS` does not change that.

Grouping repeated test sequences does not remove the known sequence overlap between training and test partitions. These intervals therefore do not establish performance on a strictly sequence-disjoint split. `first_wins` is the fraction of bootstrap draws with a positive difference, not the probability that a scientific hypothesis is true.


In [18]:
if "LucaOne, both streams" in predictions:
    comparisons = [
        (
            "LucaOne both streams minus the best BLF single encoder",
            "LucaOne, both streams",
            max(pooled, key=lambda k: scores[k]),  # Selected by observed test score.
        ),
        (
            "LucaOne both streams minus the three concatenated",
            "LucaOne, both streams",
            "BLF three encoders concatenated",
        ),
        (
            "LucaOne gene stream minus composition",
            "LucaOne, gene stream",
            "Composition (GC, GC3, codon use, length)",
        ),
    ]
    found = train.seed_averaged_bootstrap(
        "regression",
        y[test_rows],
        {k: [v] for k, v in predictions.items()},
        comparisons,
        groups=rows["sequence"].to_numpy()[test_rows],
        draws=BOOTSTRAP_DRAWS,
    )
    display(pd.DataFrame(found).set_index("comparison").round(4))

    # Publish both result tables after the paired comparisons are calculated.
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(found).to_csv(
        RESULTS_DIR / f"{BENCHMARK}_comparisons.csv", index=False,
    )
    summary.to_csv(RESULTS_DIR / f"{BENCHMARK}_probe_scores.csv")
else:
    print("Run the LucaOne cells first; this cell compares what they produce.")


,difference,ci_low,ci_high,first_wins,rows,groups
comparison,,,,,,
LucaOne both streams minus the best BLF single encoder,0.1704,0.1370,0.2037,1.0,3579,3021
LucaOne both streams minus the three concatenated,0.0924,0.0658,0.1196,1.0,3579,3021
LucaOne gene stream minus composition,0.1606,0.1302,0.1908,1.0,3579,3021


**Interpretation.** The retained comparison of concatenated representations has a positive interval for LucaOne minus BioLangFusion. That supports a difference under this fixed-split ridge comparison and its grouped test resampling. It does not resolve the training/test overlap or establish robustness across new splits. The following Phase A table changes the predictor as well as its representation handling, so it supplies context rather than another row in this matched probe comparison.


### Context from Phase A's trained token-level heads

Phase A trains a predictor over token representations, including BioLangFusion's TextCNN, whereas the table above uses ridge regression on pooled vectors. The architectures and fitting procedures differ; calling one categorically stronger would go beyond this comparison.

The reader below groups all available Phase A JSON records by variant and averages their test scores. It does not select a compatible configuration cohort or distinguish every setting, such as entropy weight. Its printed “same rows and same split” message describes the intended benchmark relationship, not a compatibility check performed by this reader. Treat this as a coarse result inventory until its record selection is reviewed.


In [19]:
phase_a = []
for path in sorted((STAGE4_DIR / "results" / "phaseA" / BENCHMARK).glob("*.json")):
    result = train.Result.load(path)
    phase_a.append({
        "variant": result.variant,
        "seed": result.config["seed"],
        "test": result.test_score,
    })

if phase_a:
    best = (
        pd.DataFrame(phase_a)
        .groupby("variant")["test"]
        .mean()
        .sort_values(ascending=False)
    )
    display(best.round(4).to_frame("Phase A, trained TextCNN"))
    print(
        "Different head, same rows and same split: not a matched comparison with the probe "
        "table, which is why it is in its own table."
    )
else:
    print("No Phase A results found for this benchmark yet.")


,"Phase A, trained TextCNN"
variant,
concat,0.5582
cross,0.5527
unimodal:rna,0.5517
mil,0.5507
unimodal-grid:rna,0.5474
unimodal:dna,0.5391
unimodal:prot,0.5311


Different head, same rows and same split: not a matched comparison with the probe table, which is why it is in its own table.


---

## Part 4: IsoFormer-style GTEx comparisons across seeds

The LucaOne analysis above holds the ridge procedure fixed while comparing representations. This section asks a different question: how much do the local GTEx modality comparisons vary when the cross-attention predictor is trained again? One transcript record supplies a genomic DNA window, transcript RNA, a protein sequence and expression targets for 30 tissues. The tissue values are outputs to predict; the modality variants select which sequence inputs are available.

The local IsoFormer-style model combines unequal-length token representations through cross-attention, without BioLangFusion's codon grid. Its encoders remain frozen. [Stage 3's modality comparison](../../docs/run-records.md#2026-10-19-stage3-isoformer-gtex-isoformers-aggregator-on-the-gtex-pilot-frozen-encoders) provides the seed-0 evidence. This block requests seeds 1 and 2 for the same pilot. Changing the seed changes initialization and the 5% validation subset drawn from training data; the test partition remains fixed.

Three intended local seeds differ from the paper's five-seed, end-to-end fine-tuning procedure. The earlier “one hour on CUDA/four on Mac” estimate lacks sufficient device and run detail to serve as a forecast. Runtime depends on sequence lengths, batch budget, hardware and training duration.

### Configure the additional runs

The execution switch is in Setup. The settings below retain the existing modality combinations and training limits. `CELL_BUDGET` limits estimated attention cells in a batch; it is not a byte or GPU-memory limit. `MAX_EPOCHS` is a ceiling, with the shared trainer's stopping rules still applying.


In [ ]:
ISOFORMER_SEEDS = (1, 2)
VARIANTS = ("dna", "rna", "prot", "dna+prot", "dna+rna", "dna+rna+prot")

MAX_BATCH = 16              # Upper bound on examples per batch.
CELL_BUDGET = 300_000_000   # Estimated attention cells, not bytes.
N_HEADS = 8                 # Cross-attention heads.
VAL_FRACTION = 0.05         # Fraction of the training partition used for validation.
MAX_EPOCHS = 500            # Maximum training epochs per fit.


In [20]:
if RUN_ISOFORMER_SEEDS:
    gtex_streams = isoformer.GTExStreams.open(CACHE_ROOT)
    for seed in ISOFORMER_SEEDS:
        print(f"\n--- seed {seed} ---")
        for variant in VARIANTS:
            made = isoformer.loaders_for(
                gtex_streams,
                batch_size=MAX_BATCH,
                val_fraction=VAL_FRACTION,
                seed=seed,
                budget=CELL_BUDGET,
                n_heads=N_HEADS,
            )
            train.run(
                gtex_streams,
                variant,
                train.TrainConfig(
                    batch_size=MAX_BATCH, max_epochs=MAX_EPOCHS, seed=seed,
                ),
                DEVICE,
                GTEX_RESULTS,
                builder=isoformer.builder,
                loaders=made,
                tag=(
                    f"gtex_pilot__{variant}__heads{N_HEADS}"
                    f"__budget{CELL_BUDGET // 1_000_000}M__seed{seed}"
                ),
                n_heads=N_HEADS,
            )
else:
    print("RUN_ISOFORMER_SEEDS is off; the reading cell below works with whatever seeds exist.")


RUN_ISOFORMER_SEEDS is off; the reading cell below works with whatever seeds exist.


### Read the reference values and local results

The reference table retains IsoFormer's published Table 2 Spearman values for each modality combination. These are context for a local frozen-encoder pilot, not target values that would establish reproduction. The [Stage 4 guide](README.md#isoformer-combining-gtex-sequence-inputs) explains the comparison boundary.

**Reader limitation.** Historical local records can use the generic variant name `IsoFormer` for different modality combinations and batch configurations. The reader below deduplicates only by variant and seed, so it can collapse distinct runs and fail to match the reference rows. Its retained one-row table does not establish a complete modality comparison or a three-seed spread. Correct record identification and cohort selection remain necessary before interpreting such a summary.


In [ ]:
ISOFORMER_TABLE2 = pd.DataFrame(
    [
        ("dna", 0.43),
        ("rna", 0.61),
        ("prot", 0.46),
        ("dna+prot", 0.52),
        ("dna+rna", 0.64),
        ("dna+rna+prot", 0.65),
    ],
    columns=["variant", "their Spearman"],
).set_index("variant")


In [ ]:
# Stage 3 holds earlier runs; this notebook writes additional runs to GTEX_RESULTS.
found = []
for folder in (STAGE3_DIR / "results" / "gtex_pilot", GTEX_RESULTS):
    for path in sorted(folder.glob("*.json")) if folder.exists() else []:
        result = train.Result.load(path)
        found.append({
            "variant": result.variant,
            "seed": result.config["seed"],
            "Spearman": result.test_score,
        })


In [21]:
if found:
    # This key does not distinguish historical generic IsoFormer configurations.
    runs = pd.DataFrame(found).drop_duplicates(["variant", "seed"])
    table = runs.groupby("variant")["Spearman"].agg(["mean", "std", "count"])
    display(table.join(ISOFORMER_TABLE2).round(3))
    if table["count"].max() > 1:
        print(f"seed spread, largest standard deviation: {table['std'].max():.3f}")
else:
    print("No IsoFormer runs found yet.")


,mean,std,count,their Spearman
variant,,,,
IsoFormer,0.439,NaN,1,NaN


---

## Findings, limitations and connection to the next comparison

The retained LucaOne results support the first question: its separately obtained gene and protein vectors, concatenated and read by ridge regression, outperform the concatenated BioLangFusion vectors in the saved stability comparison. That finding belongs to the selected checkpoints, preprocessing, supplied split and predictor. It does not demonstrate that joint pretraining caused the improvement, remove known training/test sequence overlap, or establish performance for a different prediction head.

The input-length analysis explains the chosen scope. Separate streams retain the stability inputs under the selected context rule, while this notebook does not evaluate LucaOne on GTEx. Pooling is the local comparison choice; LucaOne does produce token representations. A token-level fusion adapter or windowed GTEx comparison would be a different experiment.

The second question, variation across IsoFormer-style GTEx runs, requires correctly identified and compatible records for each modality combination and seed. The retained summary reader cannot establish that comparison because it can collapse distinct configurations. Consult the individual [Stage 3 result record](../../docs/run-records.md#2026-10-19-stage3-isoformer-gtex-isoformers-aggregator-on-the-gtex-pilot-frozen-encoders) and the [Stage 4 runbook](RUNBOOK.md#the-order) when deciding which evidence can be reused.

These comparisons contribute to Stage 4's broader question of which representations to combine and how to evaluate that choice. The remaining reporting issues—GTEx coverage's hypothetical protein lengths, Phase A cohort selection and IsoFormer record grouping—should be resolved explicitly before using those tables to support further conclusions. They are separate from new encoder extraction or model training.
